# 🏛️ ZeusDL — Agent Hermes sur Google Colab

Ce notebook transforme ton Colab en **agent Hermes** connecté à ton Mastermind Zeus.
Il peut aussi fonctionner en mode **standalone** (sans Mastermind).

## Ce que fait ce notebook
1. Installe ZeusDL depuis GitHub (toujours la dernière version)
2. Configure l'agent avec ton bot Telegram
3. Se connecte au Mastermind Zeus (ou tourne en standalone)
4. Attend et exécute les ordres reçus

## Prérequis
- Un compte BangBros premium (cookies exportés)
- Ton bot Telegram (token fourni par @BotFather)
- Optionnel : IP publique de ta machine pour le Mastermind

---
> **Note :** Le Mastermind tourne sur ta machine principale avec :
> ```
> zeusdl hermes mastermind --port 8765
> ```


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Installation (toujours depuis GitHub)      ║
# ╚══════════════════════════════════════════════════════════╝
import subprocess, sys

print('📦 Installation de ZeusDL depuis GitHub...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
     'git+https://github.com/ferelking242/zeusdl.git@main#subdirectory=zeusdl'],
    capture_output=True, text=True
)
if result.returncode == 0:
    import zeusdl
    print(f'✅ ZeusDL installé (version {zeusdl.__version__ if hasattr(zeusdl, "__version__") else "OK"})')
else:
    print('⚠️  GitHub install failed, trying PyPI...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'zeusdl'])
    print('✅ ZeusDL installé depuis PyPI')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Configuration                              ║
# ╚══════════════════════════════════════════════════════════╝

# ── Hermes Agent ────────────────────────────────────────────
AGENT_ID        = 'colab-1'          # Nom unique de cet agent
MASTER_URL      = ''                 # ex: 'http://1.2.3.4:8765' | '' = standalone
POLL_INTERVAL   = 3.0               # secondes entre chaque poll

# ── Telegram ─────────────────────────────────────────────────
BOT_TOKEN       = '7909647438:AAFUtLLCo8GOuo8yjgs0C2adGsZeNVBsopw'
DEFAULT_CHANNEL = ''                 # ex: '-1001234567890'

# ── Downloads ────────────────────────────────────────────────
OUTPUT_DIR      = '/content/downloads'
QUALITY         = '1080p'           # 360p | 480p | 720p | 1080p | best
WORKERS         = 3
COOKIES_PATH    = '/content/bangbros_cookies.txt'

# ── GitHub ───────────────────────────────────────────────────
GITHUB_TOKEN    = ''                 # ton PAT GitHub (optionnel)
GITHUB_REPO     = ''                 # ex: 'ferelking242/bangbros-collection'

# ─────────────────────────────────────────────────────────────
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Injecter la config dans ZeusDL
from zeusdl.config_manager import ZeusConfig
cfg = ZeusConfig()
cfg.set_many({
    'telegram.bot_token':      BOT_TOKEN,
    'telegram.default_channel': DEFAULT_CHANNEL,
    'defaults.quality':        QUALITY,
    'defaults.workers':        WORKERS,
    'defaults.output_dir':     OUTPUT_DIR,
    'hermes.master_url':       MASTER_URL or 'http://localhost:8765',
    'hermes.agent_id':         AGENT_ID,
    'bangbros.cookies_path':   COOKIES_PATH,
    'github.token':            GITHUB_TOKEN,
    'github.default_repo':     GITHUB_REPO,
})
print(f'✅ Config OK — agent: {AGENT_ID} | qualité: {QUALITY} | sortie: {OUTPUT_DIR}')
if MASTER_URL:
    print(f'🔗 Mastermind: {MASTER_URL}')
else:
    print('ℹ️  Mode standalone (pas de Mastermind configuré)')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Upload des cookies BangBros                ║
# ╚══════════════════════════════════════════════════════════╝
# Exécute cette cellule pour uploader ton fichier cookies.txt

from google.colab import files
print('📂 Sélectionne ton fichier cookies BangBros (format Netscape)...')
uploaded = files.upload()
for fn in uploaded:
    dest = COOKIES_PATH
    os.rename(fn, dest)
    print(f'✅ Cookies sauvegardés → {dest}')
    cfg.set('bangbros.cookies_path', dest)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Vérification de mise à jour                ║
# ╚══════════════════════════════════════════════════════════╝
from zeusdl.hermes.updater import AgentUpdater

updater = AgentUpdater()
has_update = updater.check()
if has_update:
    print('🔄 Mise à jour disponible — installation...')
    updater.update()
    print('✅ Agent mis à jour. Relance les cellules depuis le début.')
else:
    print('✅ Agent à jour.')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Démarrer l'agent Hermes                   ║
# ╚══════════════════════════════════════════════════════════╝
# L'agent se connecte au Mastermind et attend des ordres.
# En mode standalone, il écoute le bot Telegram directement.

if MASTER_URL:
    from zeusdl.hermes.agent import HermesAgent
    agent = HermesAgent(
        agent_id=AGENT_ID,
        master_url=MASTER_URL,
        poll_interval=POLL_INTERVAL,
    )
    agent.set_var('quality',           QUALITY)
    agent.set_var('workers',           str(WORKERS))
    agent.set_var('cookies',           COOKIES_PATH)
    agent.set_var('output',            OUTPUT_DIR)
    agent.set_var('telegram_token',    BOT_TOKEN)
    agent.set_var('telegram_channel',  DEFAULT_CHANNEL)
    agent.set_var('github_token',      GITHUB_TOKEN)
    print(f'🚀 Agent {AGENT_ID} connecté à {MASTER_URL}')
    print('En attente d\'ordres... (envoie des ordres depuis ton Zeus CLI)')
    print()
    print('Exemples de commandes depuis ta machine :')
    print(f'  zeusdl hermes send {AGENT_ID} "download https://site-ma.bangbros.com/scenes?addon=5971"')
    print(f'  zeusdl hermes send {AGENT_ID} "push telegram"')
    agent.start()  # bloquant
else:
    # Mode standalone — bot Telegram
    from zeusdl.telebot.bot import ZeusBot
    bot = ZeusBot(token=BOT_TOKEN)
    print(f'🤖 Bot Telegram démarré en mode standalone')
    print(f'Envoie des commandes à ton bot :')
    print(f'  /download https://site-ma.bangbros.com/scenes?addon=5971')
    print(f'  /set quality 1080p')
    print(f'  /push telegram {DEFAULT_CHANNEL}')
    bot.start()

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Download manuel (sans Mastermind)          ║
# ╚══════════════════════════════════════════════════════════╝
# Exécute un script .zeus directement dans Colab

from zeusdl.zscript.runner import ZeusScriptRunner

SCRIPT = f"""
# Script BangBros → Telegram
set quality   {QUALITY}
set workers   {WORKERS}
set output    {OUTPUT_DIR}
set cookies   {COOKIES_PATH}

download https://site-ma.bangbros.com/scenes?addon=5971

push telegram
    token    {BOT_TOKEN}
    channel  {DEFAULT_CHANNEL or 'METS_TON_CHANNEL_ID_ICI'}
    message  Batch Colab {QUALITY} terminé ✓
"""

runner = ZeusScriptRunner()
runner.run_string(SCRIPT)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Raccourcis utiles                         ║
# ╚══════════════════════════════════════════════════════════╝

# Télécharger seulement (sans push)
from zeusdl.bulk_download import BulkDownloader
dl = BulkDownloader(output_dir=OUTPUT_DIR, cookies=COOKIES_PATH,
                    workers=WORKERS, quality=QUALITY)
# dl.run('https://site-ma.bangbros.com/scenes?addon=5971')

# Uploader seulement vers Telegram
from zeusdl.telebot.uploader import TelegramUploader
up = TelegramUploader(bot_token=BOT_TOKEN, channel=DEFAULT_CHANNEL)
# up.upload_directory(OUTPUT_DIR)

# Mettre à jour l'agent
from zeusdl.hermes.updater import AgentUpdater
# AgentUpdater().update()

print('Raccourcis chargés. Décommente les lignes que tu veux exécuter.')